In [ ]:
import h5py
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import random
from typing import Optional, Tuple, List

class E2EDataset(Dataset):
    """Universal dataset for E2E CMS - handles fine-tune (labeled) & pretrain (unlabeled)"""
    
    def __init__(self, h5_path: str, mode: str = 'finetune', subset_ratio: float = 0.1, 
                 seed: int = 42, transforms=None, cache_images: bool = False):
        """
        Args:
            h5_path: Path to .h5 file
            mode: 'finetune' (images+labels) or 'pretrain' (images only)
            subset_ratio: Fraction of data to use (0.1 = 10%)
            seed: For reproducible subsampling
            transforms: torchvision transforms
            cache_images: Load subset to RAM (for speed, disable for 30GB)
        """
        super().__init__()
        self.h5_path = h5_path
        self.mode = mode
        self.transforms = transforms
        self.cache_images = cache_images
        
        # Set seed for reproducibility
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        
        # Open h5 and get info
        with h5py.File(h5_path, 'r') as f:
            if mode == 'finetune':
                assert 'images' in f and 'labels' in f, "Finetune needs 'images'+'labels'"
                self.n_samples = len(f['images'])
                self.img_shape = f['images'][0].shape
                self.n_classes = len(np.unique(f['labels'][:]))
            else:  # pretrain
                assert 'images' in f, "Pretrain needs 'images'"
                self.n_samples = len(f['images'])
                self.img_shape = f['images'][0].shape
            
            print(f"Total samples: {self.n_samples}, Image shape: {self.img_shape}")
        
        # Create unbiased stratified subset indices
        indices = self._create_subset_indices(subset_ratio)
        self.indices = np.array(indices)
        print(f"Using subset: {len(self.indices)}/{self.n_samples} samples ({subset_ratio*100:.1f}%)")
        
        # Pre-cache subset if enabled (good for finetune 500MB, not 30GB pretrain)
        if cache_images:
            print("Caching images to RAM...")
            with h5py.File(h5_path, 'r') as f:
                self.cached_images = f['images'][self.indices]
                if mode == 'finetune':
                    self.cached_labels = f['labels'][self.indices]
    
    def _create_subset_indices(self, ratio: float) -> List[int]:
        """Create stratified, unbiased subset indices"""
        with h5py.File(self.h5_path, 'r') as f:
            if self.mode == 'finetune':
                labels = f['labels'][:]
                # Stratified split by class (no bias!)
                train_idx, _ = train_test_split(
                    np.arange(len(labels)), 
                    train_size=ratio, 
                    stratify=labels, 
                    random_state=42
                )
                return train_idx.tolist()
            else:
                # Random subset for pretraining (unlabeled)
                return random.sample(range(self.n_samples), int(self.n_samples * ratio))
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        
        with h5py.File(self.h5_path, 'r') as f:
            # Memory-mapped access (no full load)
            if self.mode == 'finetune':
                image = f['images'][actual_idx]
                label = f['labels'][actual_idx]
                if self.transforms:
                    image = self.transforms(image)
                return image, label
            else:  # pretrain
                image = f['images'][actual_idx]
                if self.transforms:
                    image = self.transforms(image)
                return image  # For MAE/DiT masking

def get_dataloaders_finetune(finetune_h5: str, subset_ratio: float = 0.1, batch_size: int = 32, 
                           num_workers: int = 2, val_split: float = 0.1):
    """Complete pipeline for finetuning"""
    from torchvision import transforms
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Full dataset
    full_dataset = E2EDataset(finetune_h5, mode='finetune', subset_ratio=1.0, transforms=transform)
    
    # Train/val split (from subset)
    train_idx, val_idx = train_test_split(
        range(len(full_dataset)), test_size=val_split, 
        stratify=full_dataset.cached_labels if hasattr(full_dataset, 'cached_labels') else None
    )
    
    train_dataset = torch.utils.data.Subset(full_dataset, train_idx)
    val_dataset = torch.utils.data.Subset(full_dataset, val_idx)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                            num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                          num_workers=num_workers, pin_memory=True)
    
    return train_loader, val_loader, len(train_dataset), len(val_dataset)

def get_dataloaders_pretrain(pretrain_h5: str, subset_ratio: float = 0.05, batch_size: int = 32, 
                           num_workers: int = 2):  # Smaller subset for 30GB
    """Pipeline for pretraining (MAE/DiT)"""
    from torchvision import transforms
    
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])
    
    dataset = E2EDataset(pretrain_h5, mode='pretrain', subset_ratio=subset_ratio, transforms=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                       num_workers=num_workers, pin_memory=True)
    return loader

# Usage example for your notebook
if __name__ == "__main__":
    PATH_FINETUNE = "path1_finetune.h5"  # 5GB
    PATH_PRETRAIN = "path2_pretrain.h5"   # 30GB
    
    # Finetuning (10% = ~500MB)
    train_loader, val_loader, n_train, n_val = get_dataloaders_finetune(
        PATH_FINETUNE, subset_ratio=0.1, batch_size=32
    )
    print(f"Train: {n_train}, Val: {n_val}")
    
    # Pretraining (5% = ~1.5GB)
    pretrain_loader = get_dataloaders_pretrain(PATH_PRETRAIN, subset_ratio=0.05, batch_size=32)
    
    # Test one batch
    for images, labels in train_loader:
        print(f"Batch shape: {images.shape}, Labels: {labels.shape}")
        break